# Sales Agent Orchestrator using OpenAI Agent SDK with tools and handoff

In [ ]:
# Imports
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio 

In [ ]:
load_dotenv(override=True)

In [ ]:
#This function is just checking if sendgrind working. Verify Single Sender at sendgrid.com before executing this
# Expected output is "202" as status code in working condition
def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("cksharpa@gmail.com")  # Change to your verified sender
    to_email = To("<receiver>@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email using Sengrid")
    mail = Mail(from_email, to_email, "SendGrid : Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

In [ ]:
instructions1 = "You are a sales expert of a healthcare product company with deep clinical knowledge. \
You write a concise cold email to hospital CEO highlighting how our product improves patient outomes, \
backed by clinical evidence, efficiency gains and compliance benefits. Keep tone professional, data-driven and outcome-focused."

instructions2 = "You are a relationship-driven healthcare sales consultant. \
You write a warm, personalized cold email to healthcare organization CEO focusing on understanding their challenges, \
building trust and positioning our product as a long-term partner solution. Keep tone conversational and empathetic."

instructions3 = "You are results-oriented sales strategist. \
You write a sharp cold email to a healthcare CEO emphasizing Return on Investment(RoI), cost savings, \
operational efficiency and revenue growth enabled by our product. Keep the tone crisp and business-focused."

In [ ]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-oss-20b")

sales_agent2 = Agent(name="Empathetic Sales Agent", instructions=instructions2, model="gpt-oss-20b")

sales_agent3 = Agent(name="Business-focused Sales Agent", instructions=instructions3, model="gpt-oss-20b")

In [ ]:
@function_tool
def send_email(subject: str, body: str) -> Dict[str, str]:
    """Send email to all prospect Clients"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("cksharpa@gmail.com")  # Change to your verified sender
    to_email = To("<receiver>@gmail.com")  # Change to your recipient
    content = Content("text/html", body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
send_email

In [ ]:
#Convert all sales agents as tools
message = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=message)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=message)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=message)

tools = [tool1, tool2, tool3]

In [ ]:
tools

In [ ]:
subject_instructions = "You can write a concise subject for a cold sales email that is likely to get a response"

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which needs to convert into HTML email body with simple, clear and compelling layout."

subject_writer = Agent(name="Email Subject Writer", instructions=subject_instructions, model="gpt-oss-20b")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML converter", instructions=html_instructions, model="gpt-oss-20b")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to HTML email body")

In [ ]:
email_tools = [subject_tool, html_tool, send_email]

In [ ]:
email_tools

In [ ]:
instructions ="You are responsible for formatting and sending emails. \
Given an email body, first generate an appropriate subject using the subject_writer tool. \
Then convert the body into HTML using the html_converter tool. \
Finally, send the email using the send_html_email tool with the generated subject and HTML content."

email_agent = Agent(name="Email Manager", instructions=instructions, 
    tools=email_tools, model="gpt-oss-20b", handoff_description="Convert an email to HTML and send it")

In [ ]:
handoff=[email_agent]

In [ ]:
manager_instructions = "You are a sales manager of the healthcare product company. \
Your objective is to identify the single most effective cold sales email using the available sales agent tools. \
\
Follow this process: \
1. Generate Drafts: Use all three sales agent tools to produce three distinct email drafts. \
Ensure all drafts are generated before moving forward. \
2. Evaluate & Select: Assess the drafts and select the one that is most compelling and effective. \
You may regenerate drafts using the tools if needed. \
3. Handoff: Send only the selected email draft to the 'Email Manager' agent for formatting and delivery. \
\
Rules: \
- Always use the sales agent tools to generate drafts; do not create them manually. \
- Only one final email must be handed off to the 'Email Manager'. \
"

sales_manager = Agent(
    name="sales_manager",
    instructions=manager_instructions,
    tools=tools,
    handoffs=handoff,
    model="gpt-oss-20b"
)

manager_message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, manager_message)

# Receiver email address should have received email from sender email address (Check in spam folder too)